# Google Patents Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook searches Google Patents and exports structured patent metadata for research. Enter search terms, set a result count, and download patent titles, inventors, assignees, filing dates, snippets, and PDF links as CSV or Excel.

The notebook queries Google Patents directly — no API key required. Results include all the metadata visible on a Google Patents search results page, structured for analysis.

## Key Features

1. **No API Key Required**: Queries Google Patents directly
2. **Keyword Search**: Search across patent titles, abstracts, and descriptions
3. **Pagination**: Fetch multiple pages of results (10 per page)
4. **Rich Metadata**: Titles, patent IDs, filing dates, inventors, assignees, snippets, and PDF links
5. **HTML Cleaning**: Strips markup from titles and snippets for clean exports
6. **Structured Export**: Download results as CSV or styled Excel

## Workflow

1. **Configure**: Enter search terms and set result count
2. **Fetch**: Retrieve patent metadata from Google Patents
3. **Review**: Preview results in a table
4. **Export**: Download structured data for further analysis

## Applications

This notebook supports research involving innovation tracking, technology studies, and intellectual property analysis — mapping patent landscapes around emerging technologies, tracing the history of inventions in a field, identifying key inventors and assignees, and contextualizing technological development within broader social and cultural dynamics. Relevant for STS (Science and Technology Studies), design anthropology, business anthropology, and medical anthropology researchers examining pharmaceutical patents.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using programmatic data retrieval to support technology and innovation research. Patents are public legal documents that reflect how organizations frame and claim technical knowledge. The notebook collects and structures patent metadata but does not interpret it. Analytical judgment remains with the researcher.

## Target Audience

Designed for researchers working at the intersection of technology and culture — from STS scholars tracking innovation trajectories to business anthropologists analyzing competitive landscapes.

## Technical Approach

The notebook queries Google Patents' internal search endpoint, which returns structured JSON data. It supports keyword search with pagination. All processing runs locally in the notebook.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of notebooks supporting computational and AI-assisted approaches to anthropological research.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).



## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Anthropological Forum. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries for Google Patents retrieval, data processing, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install pandas openpyxl ipywidgets requests -q

import re
import unicodedata
import requests
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PATENTS_URL = 'https://patents.google.com/xhr/query'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://patents.google.com/',
    'X-Requested-With': 'XMLHttpRequest',
}


def clean_html(text):
    """Strip HTML tags from text."""
    if not text:
        return ''
    return re.sub(r'<[^>]+>', '', str(text))


def normalize_text(text):
    """Normalize unicode characters."""
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u2011', '-').replace('\u2013', '-').replace('\u2014', '-')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2026', '...')
    return text


def make_slug(query):
    """Create a clean filename slug."""
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30]
    return slug if slug else 'search'


def search_patents(query, num=10, page=0):
    """Search Google Patents and return results + total count."""
    q = requests.utils.quote(query)
    url = f'{PATENTS_URL}?url=q%3D{q}%26num%3D{num}%26page%3D{page}'
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    clusters = data.get('results', {}).get('cluster', [])
    total = data.get('results', {}).get('total_num_results', 0)
    if not clusters:
        return [], total
    patents = clusters[0].get('result', [])
    return patents, total


def extract_patent_data(p):
    """Extract structured data from a Google Patents result."""
    pat = p.get('patent', {})
    return {
        'title': normalize_text(clean_html(pat.get('title', ''))),
        'patent_id': pat.get('publication_number', ''),
        'filing_date': pat.get('filing_date', ''),
        'publication_date': pat.get('publication_date', ''),
        'priority_date': pat.get('priority_date', ''),
        'inventor': normalize_text(pat.get('inventor', '')),
        'assignee': normalize_text(pat.get('assignee', '')),
        'snippet': normalize_text(clean_html(pat.get('snippet', ''))),
        'language': pat.get('language', ''),
        'pdf': pat.get('pdf', ''),
        'url': f"https://patents.google.com/patent/{pat.get('publication_number', '')}" if pat.get('publication_number') else '',
    }


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    thin_border = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'),
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin_border
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = thin_border
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f4dc Ready to configure your Google Patents search!")

## Search Configuration

Configure your Google Patents search using the interactive controls below. Set your search terms, result count, and export format.

In [ ]:
# Interactive Search Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4dc Google Patents Explorer</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook searches Google Patents and exports structured patent metadata for research.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Search:</strong> Enter keywords to search patent titles, abstracts, and descriptions</li>'
instructions_html += '<li><strong>Configure:</strong> Set how many results to fetch</li>'
instructions_html += '<li><strong>Fetch:</strong> Click the button to retrieve patent data</li>'
instructions_html += '<li><strong>Export:</strong> Download as CSV or Excel</li>'
instructions_html += '</ol>'
instructions_html += '<p style="margin-top: 10px; color: #8B8C89; font-size: 0.9em;">'
instructions_html += 'Results are fetched in pages of 10. Each page takes ~1 second.</p>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

search_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f50d Search</h3>')

query_input = widgets.Text(
    value='',
    placeholder='Enter search terms (e.g., "artificial intelligence ethnography")',
    description='Keywords:',
    layout=layout,
    style=style
)

options_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Options</h3>')

max_pages_input = widgets.IntSlider(
    value=5, min=1, max=20, step=1,
    description='Pages (10/pg):',
    style=style
)

max_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Each page returns 10 results. 5 pages = up to 50 patents.</p>'
)

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(
    description='Search Patents',
    button_style='',
    layout=widgets.Layout(width='200px', margin='20px 0'),
    style={'button_color': '#6096BA'}
)

progress_label = widgets.HTML(value='')
out = widgets.Output()


def on_fetch_clicked(_):
    out.clear_output()
    progress_label.value = ''

    query = query_input.value.strip()
    if not query:
        with out:
            print("\u26a0\ufe0f Please enter search terms.")
        return

    max_pages = max_pages_input.value

    with out:
        print(f"\U0001f4dc Searching Google Patents for: {query}")
        print(f"   Pages to fetch: {max_pages} (up to {max_pages * 10} results)")
        print()

    progress_label.value = '<span style="color: #6096BA;">Fetching patents from Google Patents...</span>'

    try:
        all_rows = []
        total_available = 0

        for page in range(max_pages):
            progress_label.value = f'<span style="color: #274C77;">\u2713 Page {page+1}/{max_pages}...</span>'
            patents, total = search_patents(query, num=10, page=page)
            total_available = total

            if not patents:
                with out:
                    print(f"   No more results after page {page+1}")
                break

            for p in patents:
                all_rows.append(extract_patent_data(p))

        progress_label.value = ''

        if not all_rows:
            with out:
                print("\u26a0\ufe0f No results found. Try different search terms.")
            return

        df = pd.DataFrame(all_rows)

        with out:
            print(f"\u2713 Retrieved {len(df)} patents (of {total_available:,} available)")
            print()
            print("\U0001f4ca Preview:")
            display(df[['title', 'patent_id', 'filing_date', 'inventor', 'assignee']].head(15))

        # Export (outside widget output context)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(query)
        base = f'google_patents_{slug}_{timestamp}'
        fmt = format_input.value

        if fmt == 'xlsx':
            filepath = f'{base}.xlsx'
            df.to_excel(filepath, sheet_name='Patent Results', index=False, engine='openpyxl')
            style_excel(filepath)
        else:
            filepath = f'{base}.csv'
            df.to_csv(filepath, index=False)

        with out:
            print()
            print(f"\u2713 Saved: {filepath} ({len(df)} patents)")
            if IN_COLAB:
                colab_files.download(filepath)

    except Exception as e:
        progress_label.value = ''
        with out:
            print(f"\u274c Error: {type(e).__name__}: {e}")
            if '503' in str(e) or '429' in str(e):
                print()
                print("\U0001f6a7 Google Patents is blocking automated queries from this IP address.")
                print("   Even a few rapid searches can trigger a temporary block, and blocks are")
                print("   more common on Colab and other cloud IPs.")
                print("   Wait 10-15 minutes before retrying, fetch fewer pages, or run the notebook locally.")
            else:
                print("   If this persists, Google may have changed the Patents endpoint,")
                print("   or this IP address may be temporarily blocked. Wait and retry, or run locally.")


fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    search_header,
    query_input,
    options_header,
    max_pages_input,
    max_help,
    export_header,
    format_input,
    fetch_btn,
    progress_label,
    out,
]))